# XOR Problem — A Neural Network from Scratch (NumPy)

In `Introduction.md` we saw **why** one perceptron can never solve XOR, and that a hidden layer fixes it. In this notebook we **build that solution ourselves** — a real neural network trained with **backpropagation**, using only NumPy. No PyTorch, no library magic: every calculation happens in front of your eyes.

**Our network (2 → 2 → 1):**

![XOR network architecture](xor_architecture.svg)

- **Input layer:** 2 neurons (x₁, x₂)
- **Hidden layer:** 2 neurons, sigmoid activation
- **Output layer:** 1 neuron, sigmoid activation → output (predicted answer, between 0 and 1)

**Forward pass (data flow):**
`Z1 = X @ W1 + b1` → `H = sigmoid(Z1)` → `Z2 = H @ W2 + b2` → `output = sigmoid(Z2)`

The **4** in every shape — X is (4 × 2), H is (4 × 2), output is (4 × 1) — is the **number of samples**. We always train on all 4 XOR examples together. This is called **full-batch** training.

## Step 0 — The Data

- `x` has shape **(4, 2)**: 4 samples, 2 features each. Every row is one XOR example.
- `y` has shape **(4, 1)**: the correct answer for each row.

| Row | x₁ | x₂ | y (XOR) |
|---|---|---|---|
| 1 | 0 | 0 | 0 |
| 2 | 0 | 1 | 1 |
| 3 | 1 | 0 | 1 |
| 4 | 1 | 1 | 0 |

Rule of XOR: the output is 1 only when the two inputs are **different**.

In [1]:
import numpy as np

x = np.array([[0,0],[0,1],[1,0],[1,1]])
print(x)

y = np.array([[0],[1],[1],[0]])
print(y)

[[0 0]
 [0 1]
 [1 0]
 [1 1]]
[[0]
 [1]
 [1]
 [0]]


## Step 1 — Initialise the Weights and Biases

Before training, the network knows nothing, so we start with **random weights** and **zero biases**.

| Parameter | Shape | Meaning |
|---|---|---|
| `w1` | (2, 2) | connections Input → Hidden: 2 inputs × 2 hidden neurons |
| `b1` | (1, 2) | one bias for each hidden neuron |
| `w2` | (2, 1) | connections Hidden → Output: 2 hidden neurons × 1 output |
| `b2` | (1, 1) | one bias for the output neuron |

**Why is `b1` only (1, 2) when we have 4 samples?** NumPy **broadcasting**: the same bias row is automatically added to all 4 sample rows. One bias per neuron, shared by every sample — that is all we need.

**`np.random.seed(...)`** fixes the random numbers, so every run starts from the same weights and gives the same result. Very useful while learning and debugging.

> ⚠️ **A real lesson hiding here — the seed matters!**
> This file originally used `seed(42)`. With only 2 hidden neurons, those particular starting weights fall into a **dead trap**: the loss gets stuck at **0.125** forever, and the outputs for (0,1) and (1,1) freeze at exactly **0.5** — the network "gives up", it cannot decide. More epochs and a bigger learning rate do **not** fix it; it is a bad starting position, not a training problem.
> With `seed(1)` (used below), the same network learns XOR perfectly. Adding more hidden neurons also escapes the trap — more neurons means more chances that at least some of them start in a useful position.
> **Try it yourself later:** change the seed back to 42 and watch the loss refuse to move. This is called the **bad initialisation / local minimum** problem, and it is one of the most practical lessons in deep learning.

In [2]:
# seed -> fix the generation of random number
np.random.seed(1)
w1 = np.random.randn(2,2)
print(w1)

b1 = np.zeros((1,2))
print(b1)

w2 = np.random.randn(2,1)
b2 = np.zeros((1,1))

print(w2)
print(b2)

[[ 1.62434536 -0.61175641]
 [-0.52817175 -1.07296862]]
[[0. 0.]]
[[ 0.86540763]
 [-2.3015387 ]]
[[0.]]


## Step 2 — The Sigmoid Function

Sigmoid squashes any number into the range **(0, 1)**:

```
sigmoid(z) = 1 / (1 + e^(−z))
```

- Big positive z → output near 1; big negative z → output near 0; z = 0 → exactly 0.5.
- An output between 0 and 1 is perfect for our XOR answer (0 or 1) — read it like a confidence: 0.98 means "almost surely 1".

> 🐛 **Bug fixed:** the original line in this file was
> `return 1 + (1 / 1 + np.exp(-x))`
> Read it carefully — that is `1 + (1 + e^(−x))` = **2 + e^(−x)**, which is always **greater than 2**! A sigmoid must stay between 0 and 1, so with this bug the network could never give a correct answer, no matter how well it trained. One small typing slip — `1 + (1 /` instead of `1 / (1 +` — broke everything. The corrected version is below.

**Its derivative has a beautiful shortcut.** If `a = sigmoid(z)`, then:

```
sigmoid'(z) = a × (1 − a)
```

That is why `sigmoid_derivative` below takes the **output** `a` (not z) and simply returns `a * (1 - a)`. In backpropagation we will already have the outputs saved, so this saves us from calculating e^(−z) again.

In [3]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(a):   # a = sigmoid output
    return a * (1 - a)

## Step 3 — Training: Forward Pass + Backpropagation

One **epoch** = one full round of: predict → measure the error → fix the weights a little. We do this 10,000 times.

### Forward pass (predict)

```
hidden = sigmoid(x @ w1 + b1)        # (4,2) @ (2,2) + (1,2) -> (4,2)
output = sigmoid(hidden @ w2 + b2)   # (4,2) @ (2,1) + (1,1) -> (4,1)
error  = y - output                  # (4,1) — how wrong are we, per sample?
```

### Backward pass (learn) — this is where the transposes live

**Line 1: `d_output = error * sigmoid_derivative(output)`**
The error signal at the output layer. The `*` here is **element-wise** (not matrix multiplication): each sample's error is scaled by how sensitive its output neuron was. Shape stays (4, 1).

**Line 2: `d_hidden = (d_output @ w2.T) * sigmoid_derivative(hidden)`** ← the line this file was stuck on

The output's error must be sent **back** to the 2 hidden neurons, through the same connections the signal originally came from — `w2`. But `w2` points forward (hidden → output). To travel backward, we use it **flipped: `w2.T`**, its transpose. Shape check: `d_output` (4,1) @ `w2.T` (1,2) → (4,2) ✓ — one error value for each hidden neuron, for each sample. Without the transpose, (4,1) @ (2,1) is **mathematically impossible** — the inner dimensions 1 and 2 do not match.

**Lines 3–6: calculate each weight's gradient, then update the weight**

```
w2 += lr * (hidden.T @ d_output)                     # (2,4) @ (4,1) -> (2,1) = shape of w2 ✓
b2 += lr * np.sum(d_output, axis=0, keepdims=True)   # (1,1)
w1 += lr * (x.T @ d_hidden)                          # (2,4) @ (4,2) -> (2,2) = shape of w1 ✓
b1 += lr * np.sum(d_hidden, axis=0, keepdims=True)   # (1,2)
```

![Backpropagation matrix diagram — why the transpose?](xor_backprop_matrix.svg)

**So, why the transpose? Two reasons, one picture:**

1. **Shape law:** a weight can only be updated by a gradient of the **same shape**, because the update is element-by-element addition. `hidden` is (4,2) and `d_output` is (4,1) — the only way to combine them into (2,1) is `hidden.T @ d_output`.
2. **Meaning:** `dW[i][j] = Σ over all samples of (input_i of that sample) × (error_j of that sample)`. A weight deserves a big change when its input was large **and** its output's error was large. The transpose puts the **sample axis in the middle**, where matrix multiplication multiplies-and-adds it away — exactly this sum.

**Why `+=` and not `-=`?** Because we defined `error = y - output` (correct − predicted). This "delta" form already carries the minus sign inside it, so adding `lr × gradient` moves the weights **down** the loss slope. In the `w -= lr * grad` style you will see in other books, the gradient is defined the opposite way round. Both are the same mathematics.

**Bias updates:** a bias has no input of its own, so its gradient is simply the delta **summed over all 4 samples** (`np.sum(..., axis=0)`), with `keepdims=True` keeping the shape (1, 2) / (1, 1).

In [4]:
lr = 0.5

for epoch in range(10000):
    # forward pass
    hidden = sigmoid(x @ w1 + b1)
    output = sigmoid(hidden @ w2 + b2)

    error = y - output

    if epoch % 2000 == 0:
        loss = np.mean(error ** 2)
        print(f"epoch {epoch:5d}   loss = {loss:.6f}")

    # backward propagation
    d_output = error * sigmoid_derivative(output)
    d_hidden = (d_output @ w2.T) * sigmoid_derivative(hidden)

    # weight and bias updates
    w2 += lr * (hidden.T @ d_output)
    b2 += lr * np.sum(d_output, axis=0, keepdims=True)
    w1 += lr * (x.T @ d_hidden)
    b1 += lr * np.sum(d_hidden, axis=0, keepdims=True)

print("training finished")

epoch     0   loss = 0.256658


epoch  2000   loss = 0.003450


epoch  4000   loss = 0.000995


epoch  6000   loss = 0.000568


epoch  8000   loss = 0.000395


training finished


## Step 4 — Final Test

Training is over. Now run the trained network one last time — only the forward pass, no more learning — and compare its answers with the true XOR table.

In [5]:
hidden = sigmoid(x @ w1 + b1)
output = sigmoid(hidden @ w2 + b2)

print("raw output (confidence):")
print(np.round(output, 4))

print("\nrounded prediction:")
print(np.round(output).astype(int))

print("\ntarget (y):")
print(y)

raw output (confidence):
[[0.0155]
 [0.9837]
 [0.9833]
 [0.0206]]

rounded prediction:
[[0]
 [1]
 [1]
 [0]]

target (y):
[[0]
 [1]
 [1]
 [0]]


## Summary — what you just built

- A complete **2 → 2 → 1 neural network**, trained with **backpropagation**, in pure NumPy.
- **Forward pass** = predictions flow input → hidden → output.
- **Backward pass** = the error flows output → hidden, and every weight is corrected in proportion to its share of the mistake.
- **Transpose rule (never forget):** the gradient of a weight has the **same shape** as the weight. The transpose aligns the sample axis so it can be summed away — and it flips a weight matrix when the error has to travel in the opposite direction.
- Two real-world lessons came free: a **one-character bug** in sigmoid can silently kill a whole network, and a **bad initialisation** (seed 42 here) can trap a small network in a local minimum that no amount of extra training escapes.

**Next step:** build the same network in PyTorch — where `loss.backward()` does all of Step 3 for you. After doing it by hand once, you will know exactly what it is doing inside.